In [1]:
!pip install mistralai requests beautifulsoup4 streamlit pyngrok

INFO: pip is looking at multiple versions of opentelemetry-proto to determine which version is compatible with other requirements. This could take a while.
  Using cached opentelemetry_exporter_otlp_proto_common-1.40.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached opentelemetry_exporter_otlp_proto_http-1.40.0-py3-none-any.whl.metadata (2.5 kB)
   ---------------------------------------- 0.0/9.1 MB ? eta -:--:--
   ---------------------------------------- 0.1/9.1 MB 3.4 MB/s eta 0:00:03
   - -------------------------------------- 0.3/9.1 MB 3.2 MB/s eta 0:00:03
   -- ------------------------------------- 0.5/9.1 MB 3.6 MB/s eta 0:00:03
   -- ------------------------------------- 0.7/9.1 MB 3.8 MB/s eta 0:00:03
   --- ------------------------------------ 0.9/9.1 MB 3.9 MB/s eta 0:00:03
   ---- ----------------------------------- 1.1/9.1 MB 4.1 MB/s eta 0:00:02
   ----- ---------------------------------- 1.2/9.1 MB 4.1 MB/s eta 0:00:02
   ------ --------------------------------- 1.4/9

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-auth 2.40.2 requires cachetools<6.0,>=2.0.0, but you have cachetools 7.0.3 which is incompatible.
tensorflow-intel 2.15.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.20.3, but you have protobuf 6.33.5 which is incompatible.


In [ ]:
import os
os.environ["MISTRAL_API_KEY"] = "XXXXXXXXXXXXXXXXXXXXXXXXXXX"
os.environ["SERPAPI_API_KEY"] = "XXXXXXXXXXXXXXXXXXXXXXXXXXX"

In [3]:
import os
import json
import requests
from bs4 import BeautifulSoup
from mistralai import Mistral

mistral_api_key = os.environ["MISTRAL_API_KEY"]
serpapi_api_key = os.environ.get("SERPAPI_API_KEY", "")
mistral_model = "mistral-small-latest"
mistral_client = Mistral(api_key=mistral_api_key)

In [4]:
def call_llm(prompt: str) -> str:
    res = mistral_client.chat.complete(
        model=mistral_model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return res.choices[0].message.content

In [5]:
def make_web_research(query: str, num_results: int = 5):
    try:
        n = int(num_results)
    except:
        n = 5
    params = {
        "engine": "google",
        "q": query,
        "api_key": serpapi_api_key,
        "num": n
    }
    r = requests.get("https://serpapi.com/search.json", params=params)
    data = r.json()
    results = []
    for item in data.get("organic_results", [])[:n]:
        results.append({
            "title": item.get("title"),
            "link": item.get("link"),
            "snippet": item.get("snippet")
        })
    return results


In [6]:
def visit_web_page(url: str, max_chars: int = 6000):
    if not url.startswith("http://") and not url.startswith("https://"):
        return {"error": "invalid_url", "url": url}

    try:
        r = requests.get(url, timeout=10)
    except Exception as e:
        return {"error": "request_failed", "url": url, "details": str(e)}

    soup = BeautifulSoup(r.text, "html.parser")
    text = "\n".join(soup.stripped_strings)
    return text[:max_chars]

In [7]:
def summarize_content(text: str, user_query: str, max_chars: int = 4000):
    snippet = text[:max_chars]
    prompt = (
        "Tu es un agent de recherche de clinical guidelines.\n"
        "Question de l'utilisateur: "
        + user_query
        + "\n\nContenu de la page:\n'''"
        + snippet
        + "'''\n\nFournis un résumé structuré, en mettant en avant:\n"
        "- le type de guideline\n- la population ciblée\n- les recommandations principales\n"
        "- la source et l'année si possible."
    )
    return call_llm(prompt)

In [8]:
def save_research_note(path: str, content: str):
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)
    return "note_sauvegardee: " + path

In [9]:
def get_clinical_function_lists():
    return [
        {"name": "make_web_research", "example": 'make_web_research("hypertension clinical guidelines", 5)'},
        {"name": "visit_web_page", "example": 'visit_web_page("https://example.com")'},
        {"name": "summarize_content", "example": 'summarize_content("texte", "question utilisateur")'},
        {"name": "save_research_note", "example": 'save_research_note("resultats.txt", "contenu")'}
    ]

def get_clinical_function_map():
    return {
        "make_web_research": make_web_research,
        "visit_web_page": visit_web_page,
        "summarize_content": summarize_content,
        "save_research_note": save_research_note
    }

In [10]:
def clinical_function_calling(prompt: str, available_function):
    functions_str = ", ".join(available_function)

    full_prompt = (
        "Tu es un agent spécialisé en recherche de clinical guidelines.\n"
        "Tu peux appeler les fonctions Python suivantes: " + functions_str + ".\n"
        "RÈGLES STRICTES:\n"
        "- make_web_research doit utiliser la question clinique de l'utilisateur.\n"
        "- visit_web_page NE DOIT ÊTRE APPELÉ QUE sur un lien retourné par make_web_research.\n"
        "- N'invente JAMAIS d'URL. Pas de 'url', pas de texte vague non valide.\n"
        "- summarize_content doit utiliser uniquement du texte réel provenant de visit_web_page.\n"
        "- Tu peux appeler plusieurs fonctions dans un seul tour.\n\n"
        "Réponds STRICTEMENT avec un JSON valide de la forme:\n"
        "{\"calls\": [ {\"function_name\": \"...\", \"parameters\": [...]}, ... ]}\n"
        "sans texte autour, sans explication, sans ```.\n\n"
        "Tâche: " + prompt
    )

    raw = call_llm(full_prompt).strip()
    start = raw.find("{")
    end = raw.rfind("}")
    if start == -1 or end == -1:
        raise ValueError("Réponse LLM sans JSON : " + raw)

    raw_json = raw[start:end+1]
    data = json.loads(raw_json)

    if isinstance(data, dict) and "calls" in data:
        return data["calls"]
    if isinstance(data, dict) and "function_name" in data:
        return [data]
    if isinstance(data, list):
        return data

    raise ValueError("Format JSON inattendu: " + raw_json)

In [11]:
def execute_clinical_task(agent_prompt, available_function, user_query):
    calls = clinical_function_calling(agent_prompt, available_function)
    func_map = get_clinical_function_map()
    results = []
    for call in calls:
        name = call["function_name"]
        params = call.get("parameters", [])
        if name == "make_web_research":
            if params:
                try:
                    n = int(params[-1])
                except:
                    n = 5
            else:
                n = 5
            q = user_query + " clinical practice guidelines"
            r = make_web_research(q, n)
            results.append(r)
        elif name == "visit_web_page":
            r = func_map[name](*params)
            results.append(r)
        elif name == "summarize_content":
            if len(params) == 1:
                r = func_map[name](params[0], user_query)
            else:
                r = func_map[name](params[0], params[1])
            results.append(r)
        elif name == "save_research_note":
            r = func_map[name](*params)
            results.append(r)
        else:
            results.append("fonction inconnue: " + name)
    return results

In [12]:
def clinical_agent_execute(user_query, available_function, max_steps: int = 5):
    history = []
    for _ in range(max_steps):
        full_prompt = (
            "Question clinique de l'utilisateur: "
            + user_query
            + "\nHistorique:\n"
            + "\n".join(str(h) for h in history)
        )

        result = execute_clinical_task(full_prompt, available_function, user_query)

        flat = []
        for r in result:
            if isinstance(r, list):
                flat.extend(r)
            else:
                flat.append(r)

        if flat and all(
            isinstance(x, dict) and x.get("error") in ("invalid_url", "no_search_results")
            for x in flat
        ):
            break

        history.append(result)

    return history

In [13]:
if __name__ == "__main__":
    functions = [f["name"] for f in get_clinical_function_lists()]
    goal = (
        "Prise en charge de l'incontinence chez l'adulte, "
        "avec focus sur les clinical practice guidelines récentes."
    )

    results = clinical_agent_execute(goal, functions, max_steps=3)

    for step in results:
        for item in step:
            if isinstance(item, list):
                for doc in item:
                    if isinstance(doc, dict) and "title" in doc:
                        print("\n•", doc["title"])
                        print("  ", doc["link"])
            elif isinstance(item, str):
                print("\nRésumé :")
                print(item.strip())


• Managing adults with urinary incontinence. Clinical ...
   https://pubmed.ncbi.nlm.nih.gov/11852599/

• Managing adults with urinary incontinence. Clinical practice ...
   https://pmc.ncbi.nlm.nih.gov/articles/PMC2213931/

• Guide pour la pratique clinique sur la prise en charge ...
   https://www.researchgate.net/publication/397749964_Guide_pour_la_pratique_clinique_sur_la_prise_en_charge_therapeutique_de_l'incontinence_urinaire_d'effort_chez_la_femme_par_le_Comite_d'urologie_et_pelviperineologie_de_la_femme_CUROPF_de_l'Association

• Guide pour la pratique clinique sur la prise en charge ...
   https://doi.org/10.1016/j.fpurol.2025.08.005

• 2012 Update: Guidelines for Adult Urinary Incontinence ...
   https://cuaj.ca/index.php/journal/article/download/10/10/37
